## Objectives

- Understand the distribution of numerical and categorical variables in the cleaned dataset.
- Identify missing values, skewness, and potential outliers that may affect modelling.
- Explore the distribution of the target variable (SalePrice) and determine whether transformations (e.g. log) are needed.
- Investigate relationships between predictors and SalePrice using correlation and visual analysis.
Generate insights that guide feature engineering and model selection.

## Inputs
* Cleaned dataset: 
    - `outputs/datasets/cleaned/CleanedData.csv`

## Outputs
- Transformed TrainSet and TestSet
- List of applied transformers:
    - Ordinal categorical encoding
    - Log/Power/Box-Cox transformations
    - Winsorization (IQR)
    - Smart correlation selection


### Import Cell

In [ ]:
import os
import time
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats


from feature_engine.encoding import OneHotEncoder
from feature_engine.selection import SmartCorrelatedSelection
from feature_engine.outliers import Winsorizer
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer

sns.set(style="whitegrid")

---

## Change working directory

We need to set the current working directory to the parent folder for consistency.

In [ ]:
BASE_DIR = Path(r"C:\Users\david\Portfolio 5\heritage-housing")
os.chdir(BASE_DIR)
print("Working directory set to:", os.getcwd())

## Load Cleaned Data

We load the cleaned dataset and split it into Train and Test sets.



This notebook loads the cleaned dataset and splits it into train and test sets. This keeps the workflow consistent with the rest of the project and avoids data leakage.

In [ ]:
# Load cleaned data
data_path = BASE_DIR / "outputs/datasets/cleaned/CleanedData.csv"
df = pd.read_csv(data_path)

print("Loaded dataset shape:", df.shape)



In [ ]:
# Split dataset into train/test
TrainSet, TestSet = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Separate features and target
X_train = TrainSet.drop("SalePrice", axis=1)
y_train = TrainSet["SalePrice"]

X_test = TestSet.drop("SalePrice", axis=1)
y_test = TestSet["SalePrice"]

---

# ===============================
## Phase 1: Exploration (Analysis only)
# ===============================

## Helper Functions

These functions help check missing values and generate diagnostic plots for numeric and categorical features.

In [ ]:
def check_missing_values(df):
    """
    Print a summary of missing values per column in the dataframe.
    """
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("* No missing values found.")
    else:
        print("* Missing values found:")
        print(missing)


def diagnostic_plots(df, col):
    """
    Generate histogram, QQ plot, and boxplot for a numeric column.
    """
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    sns.histplot(df[col], kde=True, ax=axes[0])
    stats.probplot(df[col], dist="norm", plot=axes[1])
    sns.boxplot(x=df[col], ax=axes[2])
    axes[0].set_title("Histogram")
    axes[1].set_title("QQ Plot")
    axes[2].set_title("Boxplot")
    fig.suptitle(col, fontsize=18, y=1.05)
    plt.tight_layout()
    plt.show()


def diagnostic_plots_cat(df, col):
    """
    Generate a count plot for a categorical column.
    """
    plt.figure(figsize=(4,3))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(col)
    plt.xticks(rotation=90)
    plt.show()

## Identity Variables for Feature Engineering

### Phase 1: Transformation Exploration & Selection

In [ ]:
numeric_skewed = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF', 'BsmtFinSF1']

df_engineering = X_train[numeric_skewed].copy()

check_missing_values(df_engineering)

### Numerical Transformation Evaluation (Skewness Comparison)

In [ ]:
analysis = []

for col in numeric_skewed:
    row = {"Feature": col}

    row["Original Skew"] = X_train[col].skew()
    row["Log1p Skew"] = np.log1p(X_train[col]).skew()

    analysis.append(row)

skew_df = pd.DataFrame(analysis)
skew_df

### Ordinal Categorical Variables

- `KitchenQual`, `ExterQual`, `ExterCond`, `BsmtQual`, `BsmtCond`, `GarageQual`, `GarageCond`, `FireplaceQu`  
These features have a natural order, so we will encode them numerically.

### Numeric Variables for Transformation:
- `GrLivArea`, `LotArea`, `TotalBsmtSF`, `1stFlrSF`, `BsmtFinSF1`  
These features are right-skewed or have outliers and could potentially benefit from log/power/Box-Cox transformations.

### Variables for Correlation-Based Reduction:
- All numeric features  
To reduce multicollinearity and  thus keep only the most informative features.

---

In [ ]:
ordinal_vars = [
    'KitchenQual', 'ExterQual', 'ExterCond',
    'BsmtQual', 'BsmtCond',
    'GarageQual', 'GarageCond',
    'FireplaceQu'
]

boxcox_vars = ['GrLivArea', '1stFlrSF']

yeojohnson_vars = ['LotArea', 'TotalBsmtSF', 'BsmtFinSF1']

### Feature Engineering Decisions

The following transformations were identified during exploratory analysis:

- Log/Power transformations for skewed variables
- Winsorization for outlier handling
- Smart correlation selection for multicollinearity reduction
- Ordinal encoding for ordered categorical variables

These techniques informed understanding of the dataset but were not all incorporated into the final production pipeline.

### Feature Engineering Design Decisions

The following transformations were selected and included in the final feature engineering pipeline:

- Ordinal encoding for ordered categorical variables
- One-hot encoding for nominal categorical variables
- Power transformations (Box-Cox / Yeo-Johnson) for skewed features
- Winsorization for outlier handling
- Smart Correlation Selection to reduce multicollinearity

These steps were retained to improve model performance and robustness.

# ===============================
## Phase 2: Final Pipeline (Model Ready)
# ===============================

### Categorical Encoding (One-Hot)
Ordinal variables were encoded earlier.  One-hot encoding is applied here to convert any remaining categorical variables into a machine-readable format. This ensures that each category is treated independently in the modelling process.


In [ ]:
# Automatically encode all object-type categorical columns
ohe = OneHotEncoder(variables=None, drop_last=False)

X_train = ohe.fit_transform(X_train)
X_test = ohe.transform(X_test)

print("* One-hot encoding applied!")
print(X_train.shape, X_test.shape)

### Smart Correlation 

- Spearman correlation is used as the method because the dataset contains skewed numerical variables with outliers, and we are interested in monotonic relationships rather than strictly linear dependencies.
- Correlation analysis is used to identify redundant features and reduce multicollinearity before modeling.

In [ ]:
corr_sel = SmartCorrelatedSelection(
    variables=None,
    method="spearman",
    threshold=0.6,
    selection_method="variance"
)

X_train = corr_sel.fit_transform(X_train)
X_test = corr_sel.transform(X_test)

dropped_features = corr_sel.features_to_drop_
print(dropped_features)

print("* Smart correlation selection applied!")
print("Dropped features:", corr_sel.features_to_drop_)

In [ ]:
print(X_train.shape, X_test.shape)
print(list(X_train.columns) == list(X_test.columns))

## Apply Feature Engineering Transformers

### Box-Cox
Box-Cox selected for strongly right-skewed strictly positive variables. This helps to improve the regression model performance and stability.

In [ ]:
boxcox_vars = [c for c in boxcox_vars if c in X_train.columns]


# Apply Box-Cox
pt_boxcox = PowerTransformer(method="box-cox")
X_train[boxcox_vars] = pt_boxcox.fit_transform(X_train[boxcox_vars])
X_test[boxcox_vars] = pt_boxcox.transform(X_test[boxcox_vars])

### Yeo-Johnson

Yeo-Johnson selected for skewed variables (allows zeros). It improves distribution normality and helps enhance model performance.

In [ ]:
pt_yeo = PowerTransformer(method="yeo-johnson")
X_train[yeojohnson_vars] = pt_yeo.fit_transform(X_train[yeojohnson_vars])
X_test[yeojohnson_vars] = pt_yeo.transform(X_test[yeojohnson_vars])

print(X_train[boxcox_vars].min())
print("* Power transformations applied!")


### Outlier Handling (Winsorization)
Winsorization was chosen to limit the effect of extreme values while retaining all observations in the dataset. This helps improve model stability without losing potentially important data, which is important in housing price prediction problems.
Apply IQR-based Winsorization to selected numeric features.

---

In [ ]:
winsor_vars = ['GrLivArea', 'LotArea', 'TotalBsmtSF']

winsor = Winsorizer(
    capping_method='iqr',
    tail='both',
    fold=1.5,
    variables=winsor_vars
)

X_train = winsor.fit_transform(X_train)
X_test = winsor.transform(X_test)

print("* Winsorization applied!")

In [ ]:
# Evaluate Winsorization visually (Train vs Test consistency check)
for col in winsor_vars:
    diagnostic_plots(X_train, col)
    diagnostic_plots(X_test, col)

### Numerical Transformations 
- Apply Power Transformations to reduce skewness
- Box-Cox applied to strictly positive features
- Yeo-Johnson applied to all remaining skewed features 

### Transformation Summary
Final selected transformations are implemented in Phase 2 pipeline.

In [ ]:
print("=== FINAL FEATURE ENGINEERING SUMMARY ===")

print("\nOrdinal encoded variables:")
print(ordinal_vars)

print("\nBox-Cox variables:")
print(boxcox_vars)

print("\nYeo-Johnson variables:")
print(yeojohnson_vars)

print("\nWinsorized variables:")
print(winsor_vars)

print("\nCorrelation-based dropped features:")
print(corr_sel.features_to_drop_)

# ===============================
## Phase 3: Final Outputs
# ===============================

In [ ]:
# Rebuild Train/Test sets for export
TrainSet_final = X_train.copy()
TrainSet_final["SalePrice"] = y_train.values

TestSet_final = X_test.copy()
TestSet_final["SalePrice"] = y_test.values

print(TrainSet_final.shape, TestSet_final.shape)
print(y_train.shape, y_test.shape)

In [ ]:
os.makedirs("outputs/datasets/processed", exist_ok=True)
TrainSet_final.to_csv(BASE_DIR / "outputs/datasets/processed/train_processed.csv", index=False)
TestSet_final.to_csv(BASE_DIR / "outputs/datasets/processed/test_processed.csv", index=False)